<a href="https://colab.research.google.com/github/aditya01ad/colab_codes/blob/main/notebooks/OR_Prep/OR_prep_M1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pulp

# **Module 1:**

### Mathematical formulation:
> Let x = units of A, y = units of B.

* Objective: Maximize 40x + 30y

* Subject to:
  * 2x + y ≤ 100 (Machine 1)
  * x + y ≤ 80 (Machine 2)
  * x ≥ 0, y ≥ 0

## Set up PuLP model and solve:

In [15]:
from pulp import *

# Problem objective function
model = LpProblem("Product_Mix", LpMaximize)

#Disision variable
x = LpVariable("x_A", lowBound=0)
y = LpVariable("y_A", lowBound=0)

# Objective function
model += 40*x + 30*y, "Total_profit"

# Constraints
model += 2*x + y <= 100, "Machine1_hours"
model += x + y <= 80, "Machine2_hours"

# Solve
model.solve()

# Print results
print("Status:", LpStatus[model.status])
print("Optimal solution:")
print(f"  Units of A = {x.varValue}")
print(f"  Units of B = {y.varValue}")
print(f"  Max profit = ₹{value(model.objective)}")

# Print dual values
for name, c in model.constraints.items():
    print(f"  {name}: shadow price = {c.pi}, slack = {c.slack}")

Status: Optimal
Optimal solution:
  Units of A = 20.0
  Units of B = 60.0
  Max profit = ₹2600.0
  Machine1_hours: shadow price = 10.0, slack = -0.0
  Machine2_hours: shadow price = 20.0, slack = -0.0


## Swnsitivity analysis with a quick loop:

In [29]:
# Re-solve with capacity changes - icrease Machine2 capacity by 10 %
model2 = LpProblem("Product_Mix_Sens", LpMaximize)
x2 = LpVariable("x2", lowBound = 0)
y2 = LpVariable("y2", lowBound = 0)

model2 += 40*x2 + 30*y2
model2 += 2*x2 + y2 <= 100
model2 += x2 + y2 <= 88 # 80+8

model2.solve()

print("With M2 capacity = 88:")
print(f"  A = {x2.varValue}, B = {y2.varValue}, Profit = {value(model2.objective)}")


With M2 capacity = 88:
  A = 12.0, B = 76.0, Profit = 2760.0


## The transportation problem — the foundation problem:

### **Problem:**

> Two warehouse supply three stores, Warehouse capacities: WH1 = 100, WH2 = 150

> Store demands: S1 = 80, S2 = 120, S3 = 50.

> Shipping cost per unit:
WH1→S1=4,   WH1→S2=6,  WH1→S3=8,   WH2→S1=5,  WH2→S2=3,   WH2→S3=7

> Find the minimum cost shipping plan.

In [48]:
# Transportation model
warehouses = ['WH1', 'WH2']
stores = ['S1', 'S2', 'S3']
supply = {'WH1':100, 'WH2':150}
demand = {'S1':80, 'S2':120, 'S3':50}

cost = {('WH1','S1'):4, ('WH1','S2'):6, ('WH1','S3'):8, ('WH2','S1'):5, ('WH2','S2'):3, ('WH2', 'S3'):7}

# Create model
trans = LpProblem("Transfportation", LpMinimize)

# Decition variables: shipment[i][j] >= 0
ship = LpVariable.dicts("ship", (warehouses, stores), lowBound=0)

# Objective
trans += lpSum(cost[(i,j)] * ship[i][j]  for i in warehouses for j in stores)

# Supply constraints
for j in stores:
  trans += lpSum(ship[i][j] for i in warehouses) >= demand[j], f"Demand_{j}"
# Solve
trans.solve()

print("Status:", LpStatus[trans.status])
print("Optimal shipping plan:")
total_cost = 0

for i in warehouses:
  for j in stores:
    val = ship[i][j].varValue
    if val > 0:
      print(f"  {i} -> {j}: {val} units")
      total_cost += val * cost[(i,j)]
print(f"Total cost =  ₹{total_cost}")

Status: Optimal
Optimal shipping plan:
  WH1 -> S1: 80.0 units
  WH2 -> S2: 120.0 units
  WH2 -> S3: 50.0 units
Total cost =  ₹1030.0


## Adding a fixed cost — binary variables and big-M: